## 4. Model Retraining
If drift, retrain on batches 1-4, compare F1/AUC deltas.

In [ ]:

import os
import sys
project_root = r'c:\Users\anuda\Desktop\cell2cell_churn_drift'
os.chdir(project_root)
sys.path.insert(0, project_root)
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
from xgboost import XGBClassifier
from src.config import NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET, MODEL_PARAMS

# Assume has_drift from 03 (or re-run detect_drift)
has_drift = True  # for test
print(f"Drift Status: {has_drift}")

In [ ]:
from src.model.train import train_model
import pandas as pd
import joblib

if has_drift:
    # Baseline ref (on batch_0)
    baseline_results = train_model('data/batches/batch_0.csv', 'baseline')
    f1_base = baseline_results['f1']
    auc_base = baseline_results['auc']
    
    # Concat recent for retrain
    recent_data = pd.concat([pd.read_csv(f"data/batches/batch_{i}.csv") for i in range(1, 5)], ignore_index=True)
    recent_path = 'data/batches/recent_concat.csv'
    recent_data.to_csv(recent_path, index=False)
    
    # Retrain
    retrain_results = train_model(recent_path, 'retrained')
    f1_re = retrain_results['f1']
    auc_re = retrain_results['auc']
    
    # Deltas
    print(f"Delta F1: {f1_re - f1_base:+.3f} (Baseline {f1_base:.3f} ? Retrained {f1_re:.3f})")
    print(f"Delta AUC: {auc_re - auc_base:+.3f} (Baseline {auc_base:.3f} ? Retrained {auc_re:.3f})")
    
    # Save retrained
    joblib.dump(retrain_results['model'], 'models/retrained_model.pkl')
    print("Retrain complete ? adapted to drift!")
else:
    print("No drift ? skip retrain.")

In [ ]:
import datetime
version_base = f"v1_baseline_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}"
version_re = f"v2_retrained_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}"
print(f"Versions: {version_base} (F1 {f1_base:.3f}) | {version_re} (F1 {f1_re:.3f})")